# Store formats and seasons: four outcomes at once

**Multivariate Analysis of Variance** · a worked example from the StatNex teaching guide

**The story.** A fashion retailer compared three store formats in a regular and a holiday season, with five stores per combination. It tracked four outcomes at once: sales, customers, basket size and return rate.

**The question.** Do store format and season change performance as a whole, and on which measures?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/manova.csv` (30 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Store format,Season,Sales ($000),Customers (hundreds),Basket size ($),Return rate (%)
1.0,1.0,36.3,13.1,26.3,4.3
1.0,1.0,33.6,14.4,23.5,4.7
1.0,1.0,35.7,15.5,25.7,4.0
1.0,1.0,35.8,15.1,24.8,4.1
1.0,1.0,36.6,14.6,26.7,4.0
1.0,2.0,39.9,15.1,25.7,4.7
1.0,2.0,38.7,16.2,24.4,4.8
1.0,2.0,40.9,16.3,26.9,4.8
1.0,2.0,39.7,16.0,24.0,5.1
1.0,2.0,40.2,14.2,27.1,4.7
2.0,1.0,31.1,14.3,24.3,3.2
2.0,1.0,35.2,13.2,24.4,3.6
2.0,1.0,32.8,13.6,23.9,3.7
2.0,1.0,31.9,14.2,23.1,3.6
2.0,1.0,32.9,13.9,23.7,3.7
2.0,2.0,35.7,16.6,24.3,3.5
2.0,2.0,37.9,14.9,24.4,4.6
2.0,2.0,36.0,15.3,25.3,4.2
2.0,2.0,36.8,14.9,24.1,3.2
2.0,2.0,38.2,15.2,23.8,4.3
3.0,1.0,29.9,14.3,25.4,6.3
3.0,1.0,27.4,11.9,23.1,6.3
3.0,1.0,29.0,13.8,25.5,5.5
3.0,1.0,28.8,12.0,23.4,5.9
3.0,1.0,27.6,12.2,23.6,6.5
3.0,2.0,34.1,15.2,24.4,7.1
3.0,2.0,32.7,16.2,23.1,6.7
3.0,2.0,35.5,14.2,23.2,7.2
3.0,2.0,34.7,13.9,24.8,7.6
3.0,2.0,31.5,14.4,22.6,7.0

These cells save `manova_tests.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile manova_tests.py
"""MANOVA's four multivariate tests and Box's M test, as the app reports them.

statsmodels' MANOVA.mv_test() gives Pillai's trace, Wilks' lambda and Roy's largest root
with these same F approximations, but approximates Hotelling-Lawley's trace by McKeon's
(1974) rule; the app -- like R's car and SPSS -- uses the older Pillai-Samson form, which
gives a different second degrees of freedom. SciPy and statsmodels have no Box's M test.
So both are written out here, from the matrices statsmodels does compute.
"""

import numpy as np
import pandas as pd
from scipy import stats


def multivariate_tests(h, e, q, df_error):
    """Pillai, Wilks, Hotelling-Lawley and Roy for one term: value, F, df1, df2 and P.

    h is the term's hypothesis SSCP matrix, e the error SSCP matrix, q the term's degrees
    of freedom and df_error the error degrees of freedom. All four statistics are made
    from the same eigenvalues of E^-1 H; they differ in how they add them up.
    """
    p = h.shape[0]
    roots = np.clip(np.real(np.linalg.eigvals(np.linalg.solve(e, h))), 0, None)
    s, m, n = min(p, q), (abs(p - q) - 1) / 2, (df_error - p - 1) / 2

    pillai = np.sum(roots / (1 + roots))
    pillai_df1, pillai_df2 = s * (2 * m + s + 1), s * (2 * n + s + 1)
    pillai_f = (2 * n + s + 1) / (2 * m + s + 1) * pillai / (s - pillai)

    wilks = np.prod(1 / (1 + roots))
    t = np.sqrt((p * p * q * q - 4) / (p * p + q * q - 5)) if p * p + q * q - 5 > 0 else 1.0
    wilks_df1, wilks_df2 = p * q, t * (df_error - (p - q + 1) / 2) - (p * q - 2) / 2
    root = wilks ** (1 / t)
    wilks_f = (1 - root) / root * wilks_df2 / wilks_df1

    hotelling = np.sum(roots)
    hotelling_df1, hotelling_df2 = s * (2 * m + s + 1), 2 * (s * n + 1)
    hotelling_f = hotelling_df2 * hotelling / (s * s * (2 * m + s + 1))

    roy = np.max(roots)
    roy_df1 = max(p, q)
    roy_df2 = df_error - roy_df1 + q
    roy_f = roy * roy_df2 / roy_df1

    table = pd.DataFrame(
        [[pillai, pillai_f, pillai_df1, pillai_df2],
         [wilks, wilks_f, wilks_df1, wilks_df2],
         [hotelling, hotelling_f, hotelling_df1, hotelling_df2],
         [roy, roy_f, roy_df1, roy_df2]],
        index=["Pillai's trace", "Wilks' lambda", "Hotelling's trace", "Roy's largest root"],
        columns=["value", "F", "df1", "df2"],
    )
    table["P"] = stats.f.sf(table["F"], table["df1"], table["df2"])
    return table


def box_m(cells):
    """Box's M test that every cell has the same covariance matrix: (M, F, df1, df2, P).

    cells is a list of arrays, one per cell, cases down the rows and variables across.
    A cell needs more cases than variables to estimate a covariance matrix, so smaller
    cells are left out; with fewer than two cells left there is nothing to compare and
    the answer is None. F is Box's (1949) approximation, in whichever of its two forms
    the design calls for.
    """
    usable = [cell for cell in cells if cell.shape[0] > cell.shape[1]]
    if len(usable) < 2:
        return None
    p, g = usable[0].shape[1], len(usable)
    dfs = np.array([cell.shape[0] - 1 for cell in usable], dtype=float)
    covariances = [np.cov(cell, rowvar=False, ddof=1) for cell in usable]
    pooled = sum(d * c for d, c in zip(dfs, covariances)) / dfs.sum()
    m = dfs.sum() * np.linalg.slogdet(pooled)[1] - sum(
        d * np.linalg.slogdet(c)[1] for d, c in zip(dfs, covariances))

    df1 = (g - 1) * p * (p + 1) / 2
    c1 = ((1 / dfs).sum() - 1 / dfs.sum()) * (2 * p * p + 3 * p - 1) / (6 * (p + 1) * (g - 1))
    c2 = ((1 / dfs**2).sum() - 1 / dfs.sum() ** 2) * (p - 1) * (p + 2) / (6 * (g - 1))
    df2 = (df1 + 2) / abs(c2 - c1 * c1)
    if c2 > c1 * c1:
        f = m / (df1 / (1 - c1 - df1 / df2))
    else:
        f = df2 * m / (df1 * (df2 / (1 - c1 + 2 / df2) - m))
    return m, f, df1, df2, stats.f.sf(f, df1, df2)

## Libraries

In [ ]:
import itertools

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.anova import anova_lm
from manova_tests import box_m, multivariate_tests

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per observation: the response variables and one column per factor. A row
# with a blank in any of these columns is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
responses = ["Sales ($000)", "Customers (hundreds)", "Basket size ($)", "Return rate (%)"]
factors = ["Store format", "Season"]
data = data[[*responses, *factors]].dropna()
# Each factor's levels in the order they first appear in the table, as the app lists them.
for name in factors:
    data[name] = pd.Categorical(data[name], categories=pd.unique(data[name]))

## The model

In [ ]:
# Every main effect and every interaction among the factors.
order = len(factors)
terms = [term for size in range(1, order + 1) for term in itertools.combinations(factors, size)]
# C(…, Sum) is sum-to-zero coding: each effect is measured against the average of the
# levels, which is what Type III tests need. Q("…") quotes a column name.
coded = {term: ":".join(f'C(Q("{name}"), Sum)' for name in term) for term in terms}
right_side = " + ".join(coded.values())

## Descriptive statistics

In [ ]:
# For each response, one table per term of the model: each level, or each combination.
def describe(values, term):
    cells = values.groupby([data[name] for name in term], observed=True)
    table = pd.DataFrame({"n": cells.count(), "mean": cells.mean(), "SD": cells.std()})
    table["mean - SD"] = table["mean"] - table["SD"]
    table["mean + SD"] = table["mean"] + table["SD"]
    table["SE"] = table["SD"] / np.sqrt(table["n"])
    table["mean - SE"] = table["mean"] - table["SE"]
    table["mean + SE"] = table["mean"] + table["SE"]
    return table


descriptive = {response: {" * ".join(term): describe(data[response], term) for term in terms}
               for response in responses}
for response, tables in descriptive.items():
    for name, table in tables.items():
        print(f"\n{response}: {name}")
        print(table.to_string())

## Homogeneity of the covariance matrices (Box's M)

In [ ]:
# Does every cell (combination of levels) have the same covariance matrix? The test is
# very sensitive, to non-normal data as much as to unequal spreads: read a small P as a
# reason to prefer Pillai's trace below. Written out in manova_tests.py, with Box's F
# approximation, as neither SciPy nor statsmodels has it.
groups = data.groupby(factors, observed=True)
box = box_m([group[responses].to_numpy(float) for _, group in groups])
if box is None:
    print("\nBox's M: not computable (too few cells with more cases than variables)")
else:
    print(f"\nBox's M = {box[0]:.4f}, F = {box[1]:.4f}, df = ({box[2]:.0f}, {box[3]:.4f}), P = {box[4]:.4g}")

## Test for homogeneity of variance

In [ ]:
# Levene's test (deviations from the mean) for each response, across the cells.
rows = []
for response in responses:
    cells = [group[response].to_numpy(float) for _, group in groups]
    levene = stats.levene(*cells, center="mean")
    rows.append([levene.statistic, len(cells) - 1, len(data) - len(cells), levene.pvalue])
homogeneity = pd.DataFrame(rows, index=responses, columns=["F", "df1", "df2", "P"])
print()
print(homogeneity)

## Multivariate tests

In [ ]:
# All the responses tested together, each term with the others held fixed (Type III):
# could this term's means be the same on every response at once? Four statistics
# answer it; each is turned into an approximate F. Wilks' lambda is the most quoted
# (small means a large effect); Pillai's trace copes best with unequal covariance
# matrices; Roy's largest root uses only the strongest direction, and its F is an
# upper bound, so its P value is too small. They assume multivariate normal errors.
# statsmodels supplies each term's hypothesis matrix H and the error matrix E; the four
# statistics are computed from them in manova_tests.py (see there for why).
outcome = " + ".join(f'Q("{name}")' for name in responses)
manova = MANOVA.from_formula(f"{outcome} ~ {right_side}", data=data)
# Type III needs data in every combination of levels the model contains. With an empty
# cell some effects cannot be estimated and the numbers would mean nothing, so stop
# instead (the app then tests only what the data can estimate).
if np.linalg.matrix_rank(manova.exog) < manova.exog.shape[1]:
    raise SystemExit("A combination of levels in the model has no data: choose a smaller model.")
df_error = manova.exog.shape[0] - manova.exog.shape[1]
tests = manova.mv_test().results
multivariate = pd.concat({
    " * ".join(term): multivariate_tests(tests[label]["H"], tests[label]["E"],
                                         tests[label]["contrast_L"].shape[0], df_error)
    for term, label in coded.items()
})
print()
print(multivariate.to_string())

## Analysis of variance table, one response at a time

In [ ]:
# The same Type III analysis of variance as for one response, run on each in turn, to
# see where the multivariate differences lie. A significant multivariate test does not
# protect these: each is a separate test at 5%.
fits = {response: smf.ols(f'Q("{response}") ~ {right_side}', data=data).fit()
        for response in responses}
type3 = {response: anova_lm(fit, typ=3) for response, fit in fits.items()}
rows = []
for term, label in coded.items():
    for response in responses:
        line = type3[response].loc[label]
        rows.append([" * ".join(term), response, line["sum_sq"], line["df"],
                     line["sum_sq"] / line["df"], line["F"], line["PR(>F)"]])
for response, fit in fits.items():
    rows.append(["Error", response, fit.ssr, fit.df_resid, fit.ssr / fit.df_resid, np.nan, np.nan])
for response, fit in fits.items():
    rows.append(["Total", response, fit.centered_tss, len(data) - 1, np.nan, np.nan, np.nan])
anova = pd.DataFrame(rows, columns=["term", "response", "SS", "df", "MS", "F", "P"])
print()
print(anova.to_string(index=False))

## What the result says

Multivariate analysis of variance (MANOVA) tests the four outcomes together. Wilks' lambda is the share of variation left inside the groups: near 0, the groups differ a lot. The P value is the chance of a result at least this extreme if the groups did not differ.

Store format has Wilks' lambda = 0.011 and Season 0.104, both with P values far below 0.01. Their interaction has P = 0.402, so the data do not show that the season works differently in different formats.

The separate analysis of variance tables show where. Malls sell the most (\$37.7k against \$31.1k for pop-ups), and pop-ups have a return rate of 6.61% against 4.52% in malls. The holiday season lifts sales from \$32.3k to \$36.8k, but not basket size (P = 0.903).

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.